In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Download essential NLTK resources
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# ==========================================
# STEP 1: Load Financial Text Data from CSV
# ==========================================
# Load dataset from the specified path
df = pd.read_csv('/content/data.csv', encoding='latin-1')

# Note: If your CSV does not have column headers, uncomment the line below:
# df.columns = ['Sentiment', 'Sentence']

print(f"Dataset successfully loaded with {df.shape[0]} rows!")
print("\n--- First few rows ---")
print(df.head())

# ==========================================
# STEP 2: Text Preprocessing & Cleaning
# ==========================================
def clean_text(text):
    text = str(text).lower()                               # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)               # Remove special characters & numbers
    words = text.split()
    words = [w for w in words if w not in stop_words]     # Remove stop words
    return " ".join(words)

# Ensure 'Sentence' and 'Sentiment' match your CSV column names
df['Clean_Sentence'] = df['Sentence'].apply(clean_text)

# ==========================================
# STEP 3: TF-IDF Vectorization & Model Training
# ==========================================
X = df['Clean_Sentence']
y = df['Sentiment']

# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Transform text to numerical TF-IDF feature vectors
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Classifier
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

# ==========================================
# STEP 4: Evaluation & Real-Time Inference
# ==========================================
y_pred = model.predict(X_test_tfidf)

print(f"\nModel Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# Inference Function
def predict_sentiment(news_headline):
    cleaned = clean_text(news_headline)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = np.max(model.predict_proba(vec)) * 100
    return pred, prob

# Test Inference
sample_headline = "Tech giant reports record revenue, beating market estimates."
sentiment, confidence = predict_sentiment(sample_headline)
print(f"\nHeadline: '{sample_headline}'")
print(f"Predicted Sentiment: {sentiment.upper()} ({confidence:.1f}% confidence)")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset successfully loaded with 5842 rows!

--- First few rows ---
                                            Sentence Sentiment
0  The GeoSolutions technology will leverage Bene...  positive
1  $ESI on lows, down $1.50 to $2.50 BK a real po...  negative
2  For the last quarter of 2010 , Componenta 's n...  positive
3  According to the Finnish-Russian Chamber of Co...   neutral
4  The Swedish buyout firm has sold its remaining...   neutral

Model Accuracy: 69.20%

--- Classification Report ---
              precision    recall  f1-score   support

    negative       0.38      0.14      0.20       172
     neutral       0.69      0.88      0.78       626
    positive       0.75      0.63      0.69       371

    accuracy                           0.69      1169
   macro avg       0.61      0.55      0.56      1169
weighted avg       0.67      0.69      0.66      1169


Headline: 'Tech giant reports record revenue, beating market estimates.'
Predicted Sentiment: POSITIVE (43.9% confide